In [5]:
import geopandas as gpd
import ee
from shapely.geometry import Polygon, MultiPolygon

import create_grid_10_by_10
import create_grid_50_by_50
import create_grid_intersection


output_dir = "../../assets/"

## Generate full grids

In [6]:
# use Ayako's code to create the grids
grid_10km = create_grid_10_by_10.main()
grid_50km = create_grid_50_by_50.main()


/home/work/Documents/projects/2026Q2_indiamap_expansion_experiment/pm25ml/.venv/lib/python3.12/site-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/home/work/Documents/projects/2026Q2_indiamap_expansion_experiment/pm25ml/.venv/lib/python3.12/site-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


## Get material for cropping the grid for India and Bangladesh

In [7]:
# read india shapefile
grid_shapefile_path = "../../assets/grid_india_10km_shapefiles/grid_india_10km/grid_india_10km.shp"
grid_gdf = gpd.read_file(grid_shapefile_path)

### Get Bangladesh boundaries from GEE

In [8]:

def get_country_geometry(country_name='India'):
    """
    Get any country's administrative boundaries from GEE
    
    Args:
        country_name (str): Name of the country to get geometry for.
                           Must match the country_na field in LSIB dataset.
                           Examples: 'India', 'China', 'United States', 'Brazil', etc.
    
    Returns:
        ee.Geometry: Country boundary geometry from Google Earth Engine
    """
    ee.Initialize()
    # Using the Large Scale International Boundary (LSIB) dataset
    countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    
    # Filter for the specified country
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    
    # Check if the country was found
    country_count = country.size().getInfo()
    if country_count == 0:
        # Get available country names for better error message
        available_countries = countries.aggregate_array('country_na').getInfo()
        raise ValueError(f"Country '{country_name}' not found in LSIB dataset. "
                        f"Available countries include: {sorted(set(available_countries))[:10]}... "
                        f"(showing first 10 of {len(set(available_countries))} total)")
    
    return country.geometry()


# Convert GEE geometry to geopandas GeoDataFrame
def gee_geometry_to_gdf(gee_geom, target_crs='EPSG:7755'):
    """
    Convert Google Earth Engine geometry to geopandas GeoDataFrame
    and reproject to target CRS to match the grid
    
    Args:
        gee_geom: ee.Geometry object from any country
        target_crs: Target coordinate reference system (default: EPSG:7755 for India grid)
    
    Returns:
        geopandas.GeoDataFrame: Country boundaries in target CRS
    """
    # Get the geometry info from GEE
    geom_info = gee_geom.getInfo()
    geom_type = geom_info['type']
    coords = geom_info['coordinates']
    
    # Convert to shapely geometry
    if geom_type == 'Polygon':
        # Single polygon: coords = [exterior_ring, hole1, hole2, ...]
        exterior = coords[0]  # First ring is exterior
        holes = coords[1:] if len(coords) > 1 else None  # Rest are holes
        geometry = Polygon(exterior, holes)
        
    elif geom_type == 'MultiPolygon':
        # Multiple polygons: coords = [polygon1_coords, polygon2_coords, ...]
        polygons = []
        for poly_coords in coords:
            exterior = poly_coords[0]  # First ring is exterior
            holes = poly_coords[1:] if len(poly_coords) > 1 else None  # Rest are holes
            polygons.append(Polygon(exterior, holes))
        geometry = MultiPolygon(polygons)
        
    else:
        raise ValueError(f"Unsupported geometry type: {geom_type}")
    
    # Create GeoDataFrame with WGS84 CRS (EPSG:4326) 
    gdf = gpd.GeoDataFrame([{'geometry': geometry}], crs='EPSG:4326')
    
    # Reproject to target CRS to match the grid
    gdf_reprojected = gdf.to_crs(target_crs)
    
    return gdf_reprojected


In [ ]:
# Get geometry for Bangladesh
bangladesh = get_country_geometry('Bangladesh') 
bangladesh_gdf = gee_geometry_to_gdf(bangladesh, target_crs='EPSG:7755')

bangladesh_gdf.plot(facecolor='none', edgecolor='blue')

## Create new grid that contains both India and Bangladesh

In [ ]:
# Filter generated_grid to keep only cells that:
# 1. Intersect with Bangladesh, OR
# 2. Have grid_id that exists in original grid_gdf dataset

print("=== FILTERING GENERATED GRID ===")

# Get grid_ids from original grid dataset
original_grid_ids = set(grid_gdf['grid_id'])
print(f"Original grid contains {len(original_grid_ids)} grid_ids")

# Find grid cells that intersect with Bangladesh
grid_intersects_bangladesh = gpd.overlay(grid_10km, bangladesh_gdf, how='intersection')
bangladesh_grid_ids = set(grid_intersects_bangladesh['grid_id'])
print(f"Grid cells intersecting Bangladesh: {len(bangladesh_grid_ids)}")

# Find grid cells that have grid_ids in original dataset
grid_ids_in_original = set(grid_10km['grid_id']).intersection(original_grid_ids)
print(f"Grid cells with grid_ids in original dataset: {len(grid_ids_in_original)}")

# Combine both sets (union) to get final grid_ids to keep
grid_ids_to_keep = bangladesh_grid_ids.union(grid_ids_in_original)
print(f"Total unique grid_ids to keep: {len(grid_ids_to_keep)}")

# Filter the generated grid
filtered_grid = grid_10km[grid_10km['grid_id'].isin(grid_ids_to_keep)]

print(f"\nFILTER RESULTS:")
print(f"Original generated_grid size: {len(grid_10km)}")
print(f"Filtered grid size: {len(filtered_grid)}")
print(f"Dropped: {len(grid_10km) - len(filtered_grid)} cells")

# Show some sample grid_ids that were kept
print(f"\nSample grid_ids kept: {sorted(list(grid_ids_to_keep))[:10]}")

filtered_grid = filtered_grid.copy()  # Avoid SettingWithCopyWarning when working on the df after this point

=== FILTERING GENERATED GRID ===
Original grid contains 33074 grid_ids
Grid cells intersecting Bangladesh: 1565
Grid cells with grid_ids in original dataset: 33074
Total unique grid_ids to keep: 34357

FILTER RESULTS:
Original generated_grid size: 111202
Filtered grid size: 34357
Dropped: 76845 cells

Sample grid_ids kept: [1278.0, 1279.0, 1607.0, 1608.0, 1609.0, 1935.0, 1936.0, 1937.0, 1938.0, 2263.0]


In [ ]:
# Save filtered_grid as shapefiles
import os

# Create output directory

shapedir = os.path.join(output_dir, "grid_india_bangladesh_10km")
os.makedirs(shapedir, exist_ok=True)

shapefile_path = os.path.join(shapedir, f"grid_india_bangladesh_10km.shp")

# Save the filtered grid as shapefile
print(f"Saving filtered grid to: {shapefile_path}")
filtered_grid.to_file(shapefile_path)


Saving filtered grid to: ../../assets/grid_india_bangladesh_10km/grid_india_bangladesh_10km.shp


## Create file for 50 km intersection with 10 km file

In [ ]:
intersect = create_grid_intersection.main(filtered_grid, grid_50km)

intersect.to_csv(os.path.join(output_dir, 'grid_india_bangladesh_intersect_with_50km.csv'), index=False)

## Create region file

In [ ]:
# for the new grid, add a new column "k_region"
filtered_grid['k_region'] = 4

# save to parquet file
parquet_path = os.path.join(output_dir, "grid_india_bangladesh_region.parquet")
filtered_grid.to_parquet(parquet_path, index=False)